<div style="padding: 20px; background: linear-gradient(90deg, #8e2de2 0%, #4a00e0 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🔍 Module 5.1: Similarity Search Fundamentals</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">The core engine of Retrieval-Augmented Generation.</p>
</div>

---

## 1. The Core Concept

The most fundamental retrieval method is the **K-Nearest Neighbors (KNN)** approach in a vector space.
When a user asks a question, we:
1. Embed the query into a vector.
2. Calculate the distance (Cosine Similarity) between the query vector and all stored document vectors.
3. Return the `k` closest documents.

### LangChain Methods
- `similarity_search(query, k)`: Returns the raw text/metadata of the closest docs.
- `similarity_search_with_score(query, k)`: Returns the docs AND their mathematical distance score.

In [1]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# 1. Initialize free local embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create a corpus
corpus = [
    "Python is a high-level, interpreted programming language favored in AI.",
    "JavaScript is primarily used for frontend web development.",
    "Rust provides memory safety without garbage collection, great for systems.",
    "Go is designed by Google for simplicity and efficient concurrency.",
    "Java is a class-based object-oriented programming language."
]

# 3. Load into Chroma (In-memory for this demo)
docs = [Document(page_content=t, metadata={"lang": t.split()[0]}) for t in corpus]
vs = Chroma.from_documents(docs, embeddings, collection_name="lang_demo")
print("Vector Store loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector Store loaded successfully.


## 2. Standard Search vs. Scored Search
Let's query the database. Notice how we use `k=2` to restrict the output to only the top 2 results.

In [2]:
query = "Which language is best for low-level systems programming?"

# --- Standard Search ---
results = vs.similarity_search(query, k=2)
print(f"\n--- Standard Search for: '{query}' ---")
for i, d in enumerate(results):
    print(f"{i+1}. {d.page_content}")

# --- Scored Search ---
# Note: HuggingFace embeddings in Chroma return L2 (Euclidean) distance by default.
# Lower score = closer distance = better match.
scored = vs.similarity_search_with_score(query, k=2)
print("\n--- Scored Search ---")
for doc, score in scored:
    print(f"[Distance: {score:.4f}] {doc.page_content}")


--- Standard Search for: 'Which language is best for low-level systems programming?' ---
1. Python is a high-level, interpreted programming language favored in AI.
2. Java is a class-based object-oriented programming language.

--- Scored Search ---
[Distance: 0.9027] Python is a high-level, interpreted programming language favored in AI.
[Distance: 1.3135] Java is a class-based object-oriented programming language.
